# Brightline Logistics — Edge Model Trade-off Board (CST 627, Week 5)

Benchmarks small classifiers for in-cab tablets that read **scanned bills of lading** and **spoken driver notes**.
All logic lives in `scripts/`; this notebook only calls those scripts, in order.

| Step | Script |
|---|---|
| 1. Synthetic scanned documents (8 types, OCR noise, partial scans) | `generate_documents.py` |
| 2. Public driver notes (SLURP transcripts, 8 in-cab intents) | `fetch_driver_notes.py` |
| 3. Merge into one 16-class train / val / test set | `build_eval_set.py` |
| 4. Train a configuration | `train.py --config <name>` |
| 5. Benchmark it (warm-up, sync, fixed seeds, repeated trials → `results/runs.csv`) | `benchmark.py --config <name>` |
| 6. Build the Excel benchmark workbook (board, runs, training, scaling, recommendation) | `build_workbook.py` |

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ T4 GPU*.

## 0. Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("csp-cst627-week5"):
        !git clone https://github.com/nmilleson/csp-cst627-week5.git
    %cd csp-cst627-week5
    !pip install -q -r requirements.txt
else:
    # Running locally from notebooks/: work from the repo root.
    if os.path.basename(os.getcwd()) == "notebooks":
        %cd ..

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU available:', torch.cuda.is_available())"

## 1–3. Build the evaluation set

* **Synthetic documents** — bill of lading, hazmat shipping paper, delivery receipt, OS&D exception report, freight invoice,
  fuel receipt, scale ticket, lumper receipt. Headers are often cut off, ~60% are partial photos, fields are dropped/shuffled,
  and OCR noise is 0–15% per document. The **test split uses company and city names never seen in training**.
* **Driver notes** — SLURP spoken-command transcripts (CC BY 4.0), filtered to 8 intents a driver would say in the cab;
  SLURP's own train/devel/test splits are kept.

In [ ]:
!python scripts/generate_documents.py

In [ ]:
!python scripts/fetch_driver_notes.py

In [ ]:
!python scripts/build_eval_set.py

## 4–5. Baseline — 4-layer softmax-attention transformer (FP32)

In [ ]:
!python scripts/train.py --config baseline

In [ ]:
!python scripts/benchmark.py --config baseline --trials 5 --warmup 50

## Variant A — architecture: linear attention
Same 4-layer model; softmax attention is replaced by linear (elu+1) attention, which scales O(T) instead of O(T²) in sequence length.

In [ ]:
!python scripts/train.py --config linear_attention

In [ ]:
!python scripts/benchmark.py --config linear_attention --trials 5 --warmup 50

## Variant B — quantized: 4-bit NF4
No retraining: the baseline's linear layers are quantized post-training to 4-bit NF4 with bitsandbytes (FP16 matmuls). The classifier head stays FP32.

In [ ]:
!python scripts/benchmark.py --config quant_4bit --trials 5 --warmup 50

## Variant C — distilled student
A 2-layer, d=128 transformer trained on a mix of the baseline's temperature-softened predictions (α=0.7, T=2) and the hard labels.

In [ ]:
!python scripts/train.py --config distilled_student

In [ ]:
!python scripts/benchmark.py --config distilled_student --trials 5 --warmup 50

## Benchmark log (every trial of every run)

In [ ]:
import pandas as pd
runs = pd.read_csv("results/runs.csv")
runs.groupby("config", sort=False)[["accuracy", "acc_synthetic_docs", "acc_slurp_notes", "p50_ms", "p95_ms",
                                    "peak_mem_mb", "checkpoint_mb", "params_total"]].mean().round(4)

## Benchmark workbook
Builds `results/brightline_benchmark.xlsx`: **Board** (trade-off board with live formulas over every trial), **Runs**, **Training**,
**Scaling** (6ND + Chinchilla estimate for a 10x model), **Recommendation** (ranking + residual-risk register) and **Protocol**.

In [ ]:
!python scripts/build_workbook.py

In [ ]:
# Download the workbook and raw logs (Colab storage is wiped when the runtime disconnects).
if IN_COLAB:
    !zip -qr results.zip results
    from google.colab import files
    files.download("results.zip")